# avoidZoneTest v5 (260821) — 단계 6-δ: 핫스팟 개수(K) 스윕 검증

## 목적

UX 옵션으로 "짧게 / 보통 / 길게" 플로깅을 제공하는 방안이 검토 중이다. 이는 경유 핫스팟
개수(K)를 달리하는 것으로 구현되므로, K를 늘렸을 때 회피 성능과 도보 거리가 어떻게
변하는지 측정한다.

## 선행 확인 사항

**Tmap 보행자 경로 API의 경유지 상한은 5개다.** 실측(`tmapWaypointLimitTest_260731.ipynb`)
결과 0~5개는 성공하고 6개부터 10개까지 일관되게 HTTP 400(code 1100)으로 실패하였다.
따라서 K=7 옵션은 단일 호출로 구현할 수 없으며, 본 검증은 **K=3 / 4 / 5** 범위로 한정한다.

## 검증 설계

| 항목 | 내용 |
| --- | --- |
| 표본 | v4(260731)의 현실적 인접 OD 200쌍을 **그대로 재사용** |
| K 조건 | 3(v4에서 측정 완료) / 4 / 5 |
| γ | 1.0 고정 (v3에서 확정) |
| 알고리즘 | v4와 동일. 셀 1~14 무수정 계승 |

K=3은 v4(260731)에서 이미 측정하였으므로 재실행하지 않는다. 동일 표본·동일 알고리즘이므로
그대로 비교군으로 사용한다. 표본을 재생성하면 K=3 결과와 비교할 수 없으므로, 셀 15는
OD를 **생성하지 않고 로드만** 한다.

## API 호출 한도에 따른 분할 실행

Tmap 무료 사용량 한도를 고려하여 K=4와 K=5를 **다른 날에 나누어 실행**한다.
예상 호출량은 K당 약 900회이다.

### 실행 절차

| 시점 | 실행할 셀 | 설정 | 예상 호출 |
| --- | --- | --- | --- |
| **1일차** | 셀 1~14 → 셀 15 → **셀 16** | `RUN_K = 4` | 약 900회 |
| (선택) | 셀 17 | — | 0회 (K=3 vs K=4 중간 집계) |
| **2일차** | 셀 1~14 → 셀 15 → **셀 16** | `RUN_K = 5` | 약 900회 |
| 2일차 이어서 | 셀 17 → 셀 18 → 셀 19 | — | 소수 (지도용 대표 OD만) |

1일차에는 **셀 16까지 실행하고 중단**해도 된다. 셀 17은 가용한 K만 자동으로 집계하므로,
1일차에 실행하면 K=3 vs K=4 중간 결과를 볼 수 있다.

2일차에는 셀 16 상단의 `RUN_K`를 **5로 변경**한 뒤 실행한다. checkpoint 파일명에 K가
포함되므로(`step6_k4_batch0.csv`, `step6_k5_batch0.csv`) 두 실행이 서로 덮어쓰지 않는다.

### 중단 대비

배치별 checkpoint가 저장되므로, 한도 초과나 오류로 중단되어도 완료된 배치는 보존된다.
동일 `RUN_K`로 재실행하면 완료 배치를 건너뛰고 남은 배치부터 이어서 처리한다.

한 번에 처리할 배치 수를 제한하려면 셀 16의 `MAX_BATCHES_PER_RUN`을 조정한다.
기본값 8은 200 OD 전체(8배치)를 한 번에 처리하며, 4로 두면 100 OD씩 나눠 실행한다.

## 노트북을 나누지 않은 이유

K별로 노트북을 분리하면 코드가 중복되어 수정 시 양쪽을 모두 고쳐야 하고, 알고리즘이
미세하게 달라질 위험이 생긴다. 단일 노트북에서 `RUN_K` 한 값만 바꾸면 **완전히 동일한
코드**로 두 조건이 처리되므로 비교 정합성이 보장된다.

## 산출물

`output5_260821/` 폴더에 저장한다. v4의 `output4_260731/`은 읽기만 하고 수정하지 않는다.

| 파일 | 생성 시점 |
| --- | --- |
| `step6_k4_batch0~7.csv` | 1일차 (checkpoint) |
| `step6_k4_200od.csv` | 1일차 (병합) |
| `step6_k5_batch0~7.csv` | 2일차 (checkpoint) |
| `step6_k5_200od.csv` | 2일차 (병합) |
| `step6_k_sweep_summary.csv` | 셀 17 |
| `step6_k_sweep_charts.png` | 셀 18 |
| `step6_k_sweep_rep_route_*.html` | 셀 19 |

## 판단 기준

수집한 결과로 다음을 판단한다.

- **회피 성능 희생 여부**: K 증가 시 hard case 비율이 크게 오르면 해당 옵션은 안전성을
  희생한다는 의미다.
- **UX 차별화 가능 여부**: K별 평균 도보 거리 차이가 사용자가 체감할 수준인지 확인한다.
  차이가 작으면 경유 개수가 아닌 다른 축(Buffer 반경 등)으로 옵션을 설계해야 한다.
- **후보 부족 실태**: `k_eff = min(k, n)` 구조상 후보가 K보다 적으면 요청 개수를 채우지
  못한다. v4에서 중구·서구·연수의 Buffer 후보가 중앙값 6개였으므로 점검이 필요하다.


# Tmap API + 회피 알고리즘 재설계

In [ ]:
# 셀 1: 라이브러리 import 및 환경 점검
# v2 노트북 시작 — Tmap 도보 polyline 기반 회피 영역 검증

# 표준 라이브러리
import os
import json
import time
import math
import itertools
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

# HTTP·환경
import requests
from dotenv import load_dotenv

# 데이터 처리
import numpy as np
import pandas as pd

# 거리·공간 처리
from shapely.geometry import shape, Point, LineString
from shapely.ops import unary_union
from shapely.prepared import prep
from sklearn.neighbors import BallTree   # 단계 6에서 neighbor_200m 계산용

# 지도 시각화
import folium

print(f"[v2 노트북 시작 — Tmap 도보 polyline 기반]")
print(f"  pandas    {pd.__version__}")
print(f"  numpy     {np.__version__}")
print(f"  folium    {folium.__version__}")
print(f"  requests  {requests.__version__}")
print(f"  작업 디렉토리: {os.getcwd()}")

In [ ]:
# 셀 2 (수정): 인증키 로드 + 출력 디렉토리를 output2로 분리
# v1 산출물(./output)과 v2 산출물(./output2)을 명확히 구분

load_dotenv(override=True)

VWORLD_KEY = os.getenv('VWORLD_KEY')
TMAP_APP_KEY = os.getenv('TMAP_APP_KEY')

if not VWORLD_KEY:
    raise RuntimeError(".env에 VWORLD_KEY가 없다.")
if not TMAP_APP_KEY:
    raise RuntimeError(".env에 TMAP_APP_KEY가 없다.")

print(f"[인증키 로드 완료]")
print(f"  V-World:  {VWORLD_KEY[:4]}...{VWORLD_KEY[-4:]} ({len(VWORLD_KEY)}자)")
print(f"  Tmap:     {TMAP_APP_KEY[:6]}...{TMAP_APP_KEY[-4:]} ({len(TMAP_APP_KEY)}자)")

# 출력 디렉토리 — v2 전용 폴더로 분리 (v1의 ./output과 공존)
OUTPUT_DIR = Path("./output2")
OUTPUT_DIR.mkdir(exist_ok=True)
print(f"  출력 디렉토리: {OUTPUT_DIR.resolve()}")
print(f"  (v1 산출물은 ./output에 그대로 보존)")

In [ ]:
# 셀 3: v1 회피 영역 GeoJSON 로드 + shapely 변환
# v1 노트북에서 V-World로 수집한 147개 폴리곤을 그대로 활용 (시간 절약)

from shapely.geometry import shape
from shapely.ops import unary_union
from shapely.prepared import prep

# v1 산출물 경로 (output 폴더에 있음)
GEOJSON_PATH = Path("./output/incheon_avoid_zones_v1_260601.geojson")

if not GEOJSON_PATH.exists():
    raise FileNotFoundError(
        f"v1 GeoJSON 파일이 없다: {GEOJSON_PATH.resolve()}\n"
        f"v1 노트북의 셀 11까지 실행한 산출물이 있어야 한다."
    )

with open(GEOJSON_PATH, 'r', encoding='utf-8') as f:
    avoid_geojson = json.load(f)

# 147개 features → shapely 폴리곤 + 메타데이터
avoid_polygons = []
for feature in avoid_geojson['features']:
    props = feature['properties']
    geom = shape(feature['geometry'])
    avoid_polygons.append({
        'geom':       geom,
        'uqa_code':   props.get('uqa_code'),
        'uname':      props.get('uname'),
        'sigg_name':  props.get('sigg_name'),
        'feature_id': props.get('feature_id'),
    })

# 전체 폴리곤 union (교차 검사·페널티 계산용)
avoid_union = unary_union([p['geom'] for p in avoid_polygons])
avoid_union_prepared = prep(avoid_union)  # 점-폴리곤 검사 가속

# UQA 코드별 union (분석용)
avoid_union_by_code = {}
for code in ['UQA310', 'UQA320', 'UQA330']:
    geoms = [p['geom'] for p in avoid_polygons if p['uqa_code'] == code]
    if geoms:
        avoid_union_by_code[code] = unary_union(geoms)

# 통계 출력
print(f"[회피 영역 로드 완료]")
print(f"  총 폴리곤: {len(avoid_polygons)}개")
print(f"  UQA 코드별:")
for code in ['UQA310', 'UQA320', 'UQA330']:
    n = sum(1 for p in avoid_polygons if p['uqa_code'] == code)
    print(f"    {code}: {n}개")

print(f"\n  군구별:")
sigg_counts = Counter(p['sigg_name'] for p in avoid_polygons)
for sigg, n in sigg_counts.most_common():
    print(f"    {sigg:<15} {n}개")

print(f"\n  avoid_union bounds (lon_min, lat_min, lon_max, lat_max):")
print(f"    {avoid_union.bounds}")

# === v4(260731) 추가: sigg_name 원본 형식 확인 + 군구 매핑 검증 ===
# v4(260729)에서 '동구'가 '남동구'의 부분 문자열이라는 이유로 남동구 폴리곤이
# 동구로 오분류되었다. 그 결과 남동구가 표본에서 누락되고 동구 표본이 오염되었다.
# 부분 문자열 충돌을 막기 위해 긴 이름을 우선 검사하는 매핑 함수를 정의한다.

print("\n[sigg_name 원본 형식]")
for s in sorted(sigg_counts):
    print(f"    {repr(s)}")

DISTRICT_MATCH_ORDER = [
    ('michuhol', ['미추홀구', '미추홀']),
    ('namdong',  ['남동구',  '남동']),
    ('bupyeong', ['부평구',  '부평']),
    ('gyeyang',  ['계양구',  '계양']),
    ('yeonsu',   ['연수구',  '연수']),
    ('ganghwa',  ['강화군',  '강화']),
    ('seogu',    ['서구']),
    ('junggu',   ['중구']),
    ('donggu',   ['동구']),     # '남동구'보다 반드시 뒤에서 검사되어야 한다
]

def match_district(sigg_name):
    """sigg_name 문자열을 군구 영문 키로 변환한다 (부분 문자열 충돌 방지 순서)."""
    s = str(sigg_name)
    for d_eng, keys in DISTRICT_MATCH_ORDER:
        for k in keys:
            if k in s:
                return d_eng
    return None

print("\n[군구 매핑 검증]")
_map_counts = Counter()
_unmatched = []
for p in avoid_polygons:
    d = match_district(p['sigg_name'])
    if d is None:
        _unmatched.append(str(p['sigg_name']))
    else:
        _map_counts[d] += 1
for d, n in sorted(_map_counts.items(), key=lambda kv: -kv[1]):
    print(f"    {d:<10} {n}개")
print(f"    매칭 실패: {len(_unmatched)}개" + (f" {sorted(set(_unmatched))}" if _unmatched else ""))
if 'namdong' in _map_counts:
    print(f"    남동구 매핑 정상 ({_map_counts['namdong']}개) — v4_260729 오분류 해소")
else:
    print("    [경고] 남동구 폴리곤이 매핑되지 않았다. DISTRICT_MATCH_ORDER 점검 필요.")


In [ ]:
# 셀 4: 핫스팟 CSV 로드 + neighbor_200m + S_i 계산
# routeTest_v3 셀 3·5·6 이식 (핸드오프 v4 알고리즘 사양 그대로)

import ast
from sklearn.neighbors import BallTree

# === 1. CSV 로드 ===
CSV_PATH = Path("./hotspots_CCTV_20260510.csv")
if not CSV_PATH.exists():
    raise FileNotFoundError(f"핫스팟 CSV가 없다: {CSV_PATH.resolve()}")

hotspots = pd.read_csv(CSV_PATH, encoding='utf-8-sig')

# list 필드 복원 (문자열 → list)
def parse_list_field(value):
    if pd.isna(value):
        return None
    if isinstance(value, str):
        try:
            return ast.literal_eval(value)
        except (ValueError, SyntaxError):
            return None
    return value

hotspots['install_years']      = hotspots['install_years'].apply(parse_list_field)
hotspots['relocation_history'] = hotspots['relocation_history'].apply(parse_list_field)

print(f"[핫스팟 로드] {len(hotspots):,}곳, {len(hotspots.columns)}컬럼")

# === 2. neighbor_200m 계산 (BallTree) ===
# 핸드오프 D2-2 공식: S_i = cctv_count + 0.5 × neighbor_200m
EARTH_RADIUS_KM = 6371.0

t0 = time.time()
coords_rad = np.radians(hotspots[['latitude', 'longitude']].values)
tree = BallTree(coords_rad, metric='haversine')
radius_rad = 0.2 / EARTH_RADIUS_KM   # 200m → 라디안

# query_radius는 자기 자신 포함 카운트 반환 → -1 보정
counts = tree.query_radius(coords_rad, r=radius_rad, count_only=True)
hotspots['neighbor_200m'] = counts - 1

print(f"\n[neighbor_200m 계산] 소요 {time.time() - t0:.2f}초")
print(f"  분포: min {hotspots['neighbor_200m'].min()}, "
      f"max {hotspots['neighbor_200m'].max()}, "
      f"평균 {hotspots['neighbor_200m'].mean():.2f}")

n_with_neighbor = (hotspots['neighbor_200m'] > 0).sum()
print(f"  200m 내 이웃 보유: {n_with_neighbor:,}곳 ({n_with_neighbor/len(hotspots)*100:.1f}%)")

# === 3. S_i 계산 ===
hotspots['S_i'] = hotspots['cctv_count'] + 0.5 * hotspots['neighbor_200m']

print(f"\n[S_i 계산 완료]")
print(f"  분포: min {hotspots['S_i'].min():.2f}, "
      f"max {hotspots['S_i'].max():.2f}, "
      f"평균 {hotspots['S_i'].mean():.2f}")

print(f"\n  군구별 S_i 평균:")
for sigg, mean_si in hotspots.groupby('source_district')['S_i'].mean().items():
    print(f"    {sigg:<12} {mean_si:.2f}")

# === 4. 회피 영역 내부 핫스팟 표시 (방법 C에서 활용) ===
# 각 핫스팟이 회피 영역 내부에 있는지 미리 계산
def is_in_avoid_zone(row):
    pt = Point(row['longitude'], row['latitude'])
    return avoid_union_prepared.contains(pt)

t0 = time.time()
hotspots['in_avoid_zone'] = hotspots.apply(is_in_avoid_zone, axis=1)
print(f"\n[회피 영역 내부 검사] 소요 {time.time() - t0:.2f}초")
print(f"  회피 영역 내부 핫스팟: {hotspots['in_avoid_zone'].sum()}/{len(hotspots):,} "
      f"({hotspots['in_avoid_zone'].sum()/len(hotspots)*100:.2f}%)")

In [ ]:
# 셀 5: 거리·Buffer 유틸 함수 + OD 9쌍 + 후보 추출 검증
# haversine 패키지 의존 제거 — numpy 기반 직접 구현 (수학적으로 동일)

# === 0. Haversine 직접 구현 ===
def haversine_km(p_a, p_b):
    """두 (lat, lon) 좌표 사이의 Haversine 거리 (km)
    
    haversine 패키지의 hs.haversine(p_a, p_b, unit=hs.Unit.KILOMETERS)와 동일 결과.
    지구 반경 6371.0 km 사용.
    """
    lat1, lon1 = p_a
    lat2, lon2 = p_b
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    return EARTH_RADIUS_KM * c   # EARTH_RADIUS_KM은 셀 4에서 6371.0으로 정의됨

# === 1. 도보·Buffer 유틸 함수 ===
DETOUR_RATIO = 1.3   # 한국 도시 도보 환경 평균 보정계수 (핸드오프 D3)

def walking_distance_km(p_a, p_b):
    """추정 도보 거리 (km) = Haversine × 1.3"""
    return haversine_km(p_a, p_b) * DETOUR_RATIO

def detour_distance_km(origin, hotspot, destination):
    """핫스팟 단독 경유 시 우회 거리: D_i = d(O,h) + d(h,D) - d(O,D)"""
    d_oh = walking_distance_km(origin, hotspot)
    d_hd = walking_distance_km(hotspot, destination)
    d_od = walking_distance_km(origin, destination)
    return max(d_oh + d_hd - d_od, 0.0)

def get_buffer_radius(district):
    """군구별 buffer 반경 (km) — 강화군 2.5km, 그 외 1.5km (핸드오프 D4)"""
    return 2.5 if district == 'ganghwa' else 1.5

def filter_candidates_by_ellipse_buffer(df, origin, destination, buffer_r_km):
    """타원 buffer 필터링: d(O,h) + d(h,D) - d(O,D) < buffer_r × 1.5"""
    d_od = walking_distance_km(origin, destination)
    threshold = buffer_r_km * 1.5
    
    coords = df[['latitude', 'longitude']].values
    d_oh = np.array([walking_distance_km(origin, tuple(c)) for c in coords])
    d_hd = np.array([walking_distance_km(tuple(c), destination) for c in coords])
    
    mask = (d_oh + d_hd - d_od) < threshold
    return df[mask].copy()

# === 2. OD 9쌍 좌표 (routeTest_v3 OD_CANDIDATES_V2와 동일) ===
OD_CANDIDATES_V2 = {
    'ganghwa':  {'O': (37.706505, 126.443398), 'D': (37.732838, 126.517697)},
    'junggu':   {'O': (37.473565, 126.621778), 'D': (37.474748, 126.603258)},
    'donggu':   {'O': (37.470733, 126.641914), 'D': (37.487780, 126.621754)},
    'michuhol': {'O': (37.455015, 126.667435), 'D': (37.435062, 126.690721)},
    'seogu':    {'O': (37.545115, 126.676062), 'D': (37.552748, 126.650239)},
    'yeonsu':   {'O': (37.391182, 126.645092), 'D': (37.418275, 126.661122)},
    'namdong':  {'O': (37.447245, 126.731453), 'D': (37.447769, 126.700089)},
    'bupyeong': {'O': (37.507046, 126.721858), 'D': (37.482179, 126.703038)},
    'gyeyang':  {'O': (37.537367, 126.737744), 'D': (37.530279, 126.722516)},
}

# === 3. 각 OD에 대해 Buffer 후보 추출 + 표본 편향 검증 ===
print(f"[Buffer 후보 추출 검증]")
print(f"{'군구':<10} {'직선(km)':>9} {'buffer(km)':>11} {'후보수':>7} {'회피_내부':>9}")
print("-" * 55)

od_buffer_results = {}
for district, od in OD_CANDIDATES_V2.items():
    origin = od['O']
    destination = od['D']
    
    direct_km = haversine_km(origin, destination)
    buffer_r = get_buffer_radius(district)
    
    district_df = hotspots[hotspots['source_district'] == district]
    cands = filter_candidates_by_ellipse_buffer(district_df, origin, destination, buffer_r)
    n_avoid_in_buffer = int(cands['in_avoid_zone'].sum())
    
    od_buffer_results[district] = {
        'origin': origin,
        'destination': destination,
        'direct_km': direct_km,
        'buffer_r_km': buffer_r,
        'candidates': cands,
        'n_candidates': len(cands),
        'n_avoid_in_buffer': n_avoid_in_buffer,
    }
    
    print(f"{district:<10} {direct_km:>7.2f}   {buffer_r:>9}   {len(cands):>5}   "
          f"{n_avoid_in_buffer:>6}")

print(f"\n[중요 — 표본 편향 가설 검증]")
total_avoid_in_buffer = sum(r['n_avoid_in_buffer'] for r in od_buffer_results.values())
print(f"  9 OD의 Buffer 안 회피 핫스팟 총합: {total_avoid_in_buffer}곳")
if total_avoid_in_buffer == 0:
    print(f"  → 표본 편향 가설 정량 입증: Buffer 안에 회피 핫스팟이 0건이므로")
    print(f"    옵션 X 0건은 알고리즘 견고성이 아닌 OD 선정 편향의 결과")
elif total_avoid_in_buffer < 5:
    print(f"  → 표본 편향 가설 부분 입증: Buffer 안 회피 핫스팟이 매우 적음")
else:
    print(f"  → 알고리즘이 일부 회피 효과 보일 가능성 (S_i 정렬에서 탈락 등)")

## 정적 회피 페널티 함수 (방법 C 핵심)

In [ ]:
# 셀 6: 정적 회피 페널티 함수 (방법 C 핵심)
# Tmap 호출 없이 shapely 직선 교차로 회피 페널티 계산
# 위경도 1도 ≈ 111km (위도 37도 부근) 근사 사용

def compute_avoid_penalty(hotspot_lat, hotspot_lon, origin, destination,
                          avoid_union, avoid_union_prepared):
    """
    핫스팟의 정적 회피 페널티 계산.
    
    구성 요소:
    1. 핫스팟이 회피 영역 내부면 매우 큰 페널티 (999) — 옵션 X 완전 차단
    2. O → 핫스팟 직선의 회피 영역 통과 길이
    3. 핫스팟 → D 직선의 회피 영역 통과 길이
    
    단위: km (다른 항과 일치)
    호출 비용: O(1), 약 1~2 밀리초
    
    Args:
        hotspot_lat, hotspot_lon: 핫스팟 좌표
        origin: (lat, lon) 출발지
        destination: (lat, lon) 도착지
        avoid_union: shapely MultiPolygon (회피 영역 union)
        avoid_union_prepared: prep된 union (점-폴리곤 검사 가속)
    
    Returns:
        float: 페널티 (km)
    """
    # 1. 회피 영역 내부 검사 (옵션 X 차단)
    hotspot_point = Point(hotspot_lon, hotspot_lat)
    if avoid_union_prepared.contains(hotspot_point):
        return 999.0
    
    # 2. O → 핫스팟 직선의 회피 영역 통과
    line_o = LineString([
        (origin[1], origin[0]),
        (hotspot_lon, hotspot_lat),
    ])
    inter_o = line_o.intersection(avoid_union)
    len_o_deg = inter_o.length if not inter_o.is_empty else 0
    
    # 3. 핫스팟 → D 직선의 회피 영역 통과
    line_d = LineString([
        (hotspot_lon, hotspot_lat),
        (destination[1], destination[0]),
    ])
    inter_d = line_d.intersection(avoid_union)
    len_d_deg = inter_d.length if not inter_d.is_empty else 0
    
    # 위경도 → km 근사 (위도 37도 부근 1도 ≈ 111km)
    return (len_o_deg + len_d_deg) * 111.0


# === 동작 검증 — donggu·gyeyang의 회피 후보 vs 일반 후보 페널티 비교 ===
print("[정적 회피 페널티 함수 검증]")
print("=" * 70)

for district in ['donggu', 'gyeyang', 'michuhol']:   # 회피 7곳·7곳·0곳 비교
    od = OD_CANDIDATES_V2[district]
    cands = od_buffer_results[district]['candidates']
    
    # 회피 영역 내부 후보의 페널티
    avoid_cands = cands[cands['in_avoid_zone']]
    
    print(f"\n[{district}] OD: O{od['O']} → D{od['D']}")
    print(f"  Buffer 후보 {len(cands)}곳 중 회피 내부 {len(avoid_cands)}곳")
    
    if len(avoid_cands) > 0:
        print(f"\n  회피 영역 내부 후보 (S_i 상위 3개):")
        for _, row in avoid_cands.nlargest(3, 'S_i').iterrows():
            penalty = compute_avoid_penalty(
                row['latitude'], row['longitude'],
                od['O'], od['D'],
                avoid_union, avoid_union_prepared,
            )
            print(f"    {row['hotspot_id']}: S_i={row['S_i']:.2f}, "
                  f"penalty={penalty:.2f}")
    
    # 비교: 회피 영역 밖 후보 상위 3
    non_avoid = cands[~cands['in_avoid_zone']]
    print(f"\n  회피 영역 밖 후보 (S_i 상위 3개):")
    for _, row in non_avoid.nlargest(3, 'S_i').iterrows():
        penalty = compute_avoid_penalty(
            row['latitude'], row['longitude'],
            od['O'], od['D'],
            avoid_union, avoid_union_prepared,
        )
        print(f"    {row['hotspot_id']}: S_i={row['S_i']:.2f}, "
              f"penalty={penalty:.2f}")

In [ ]:
# 셀 7: Score 공식 통합 — 방법 C 적용
# 새 공식: Score_i = α·S_i - β·D_i - γ·avoid_penalty_i

# === 하이퍼파라미터 ===
ALPHA = 0.5   # S_i 가중치 (핸드오프 D6 그대로)
BETA  = 0.5   # D_i 가중치 (핸드오프 D6 그대로)
GAMMA = 1.0   # avoid_penalty 가중치 (방법 C 신규, 단계 6 후반 튜닝 대상)

# === 회피 페널티 통합 Score 함수 (방법 C) ===
def compute_scores_with_avoidance(df, origin, destination,
                                   avoid_union, avoid_union_prepared,
                                   alpha=ALPHA, beta=BETA, gamma=GAMMA):
    """방법 C 적용 Score 계산.
    Score_i = α·S_i - β·D_i - γ·avoid_penalty_i
    """
    out = df.copy()
    out['D_i'] = out.apply(
        lambda r: detour_distance_km(origin, (r['latitude'], r['longitude']), destination),
        axis=1
    )
    out['avoid_penalty_i'] = out.apply(
        lambda r: compute_avoid_penalty(
            r['latitude'], r['longitude'],
            origin, destination,
            avoid_union, avoid_union_prepared,
        ),
        axis=1
    )
    out['Score_i'] = alpha * out['S_i'] - beta * out['D_i'] - gamma * out['avoid_penalty_i']
    return out

# === 원본 Score 함수 (방법 C 적용 전, 회귀 검증용) ===
def compute_scores_original(df, origin, destination, alpha=ALPHA, beta=BETA):
    """routeTest_v3 원본 Score (회귀 검증용)."""
    out = df.copy()
    out['D_i'] = out.apply(
        lambda r: detour_distance_km(origin, (r['latitude'], r['longitude']), destination),
        axis=1
    )
    out['avoid_penalty_i'] = 0.0   # 없음 (호환성)
    out['Score_i'] = alpha * out['S_i'] - beta * out['D_i']
    return out

print(f"[Score 공식 정의 완료]")
print(f"  원본: Score_i = α·S_i - β·D_i")
print(f"  방법 C: Score_i = α·S_i - β·D_i - γ·avoid_penalty_i")
print(f"  가중치: α={ALPHA}, β={BETA}, γ={GAMMA} (초기값)")
print(f"  γ 튜닝은 단계 6-D에서 진행 (0.5, 1.0, 2.0, 5.0 비교)")

In [ ]:
# 셀 8: Greedy 알고리즘 + 방법 C vs 원본 비교
# routeTest_v3의 algorithm_greedy 이식 + 회피 페널티 적용 버전 추가

K = 3
BF_TOP_N = 15

def algorithm_greedy_v2(candidates_df, origin, destination,
                         avoid_union, avoid_union_prepared,
                         alpha=ALPHA, beta=BETA, gamma=GAMMA, k=K):
    """방법 C 적용 Greedy.
    1. S_i 상위 BF_TOP_N=15 풀 (원본과 동일)
    2. compute_scores_with_avoidance로 Score_i 계산 (회피 페널티 포함)
    3. Score_i 상위 K=3 선택
    """
    t0 = time.time()
    n = len(candidates_df)
    if n == 0:
        return pd.DataFrame(), 0
    k_eff = min(k, n)
    
    pool = candidates_df.nlargest(min(BF_TOP_N, n), 'S_i')
    scored = compute_scores_with_avoidance(
        pool, origin, destination, avoid_union, avoid_union_prepared,
        alpha=alpha, beta=beta, gamma=gamma,
    )
    selected = scored.nlargest(k_eff, 'Score_i')
    return selected, (time.time() - t0) * 1000

def algorithm_greedy_original(candidates_df, origin, destination,
                               alpha=ALPHA, beta=BETA, k=K):
    """routeTest_v3 원본 Greedy (회귀 검증용)."""
    t0 = time.time()
    n = len(candidates_df)
    if n == 0:
        return pd.DataFrame(), 0
    k_eff = min(k, n)
    
    pool = candidates_df.nlargest(min(BF_TOP_N, n), 'S_i')
    scored = compute_scores_original(pool, origin, destination, alpha=alpha, beta=beta)
    selected = scored.nlargest(k_eff, 'Score_i')
    return selected, (time.time() - t0) * 1000


# === 9 OD에 대해 원본 vs 방법 C 직접 비교 ===
print("=" * 80)
print("[Greedy 원본 vs 방법 C 비교 — 9 OD]")
print("=" * 80)

comparison_results = []
for district, od in OD_CANDIDATES_V2.items():
    cands = od_buffer_results[district]['candidates']
    
    # 원본
    sel_orig, _ = algorithm_greedy_original(cands, od['O'], od['D'])
    orig_ids = set(sel_orig['hotspot_id'])
    orig_avoid_count = int(sel_orig['in_avoid_zone'].sum()) if 'in_avoid_zone' in sel_orig.columns else 0
    
    # 방법 C
    sel_v2, _ = algorithm_greedy_v2(cands, od['O'], od['D'], avoid_union, avoid_union_prepared)
    v2_ids = set(sel_v2['hotspot_id'])
    v2_avoid_count = int(sel_v2['in_avoid_zone'].sum()) if 'in_avoid_zone' in sel_v2.columns else 0
    
    # K=3 일치 여부
    if orig_ids == v2_ids:
        diff_status = "동일"
    else:
        diff_status = f"다름 (원본만: {len(orig_ids - v2_ids)}, v2만: {len(v2_ids - orig_ids)})"
    
    # 방법 C의 페널티 합계
    v2_total_penalty = sel_v2['avoid_penalty_i'].sum()
    
    comparison_results.append({
        'district': district,
        'orig_avoid_in_K3': orig_avoid_count,
        'v2_avoid_in_K3': v2_avoid_count,
        'diff_status': diff_status,
        'v2_total_penalty': v2_total_penalty,
    })
    
    print(f"\n[{district}]")
    print(f"  원본 K=3 회피 영역 내부: {orig_avoid_count}개")
    print(f"  방법 C K=3 회피 영역 내부: {v2_avoid_count}개")
    print(f"  K=3 비교: {diff_status}")
    print(f"  방법 C 총 페널티: {v2_total_penalty:.2f}")

# 종합
print(f"\n{'=' * 80}")
print(f"[종합]")
df_compare = pd.DataFrame(comparison_results)
n_diff = sum(1 for r in comparison_results if "다름" in r['diff_status'])
print(f"  K=3 선정이 다른 OD: {n_diff}/9")
print(f"  원본 K=3 회피 내부 총: {df_compare['orig_avoid_in_K3'].sum()}")
print(f"  방법 C K=3 회피 내부 총: {df_compare['v2_avoid_in_K3'].sum()} (페널티 999로 차단)")

### Tmap 보행자 호출 + polyline 교차 검사

In [ ]:
# 셀 9: Tmap 보행자 호출 함수 + polyline 교차 검사
# v1 노트북의 셀 21을 v2에 이식 + 교차 검사 함수 추가

TMAP_PEDESTRIAN_URL = "https://apis.openapi.sk.com/tmap/routes/pedestrian?version=1"

def get_pedestrian_route(origin, destination, waypoints=None, debug=False):
    """Tmap 보행자 경로안내 API 호출.
    
    Args:
        origin, destination: (lat, lon)
        waypoints: [(lat, lon), ...] 또는 None (최대 5개)
    Returns:
        dict: path([(lat, lon), ...]), distance_m, duration_ms, status
    """
    headers = {
        "Accept": "application/json",
        "Content-Type": "application/json",
        "appKey": TMAP_APP_KEY,
    }
    payload = {
        "startX": origin[1], "startY": origin[0],
        "endX": destination[1], "endY": destination[0],
        "startName": "출발지", "endName": "도착지",
        "reqCoordType": "WGS84GEO", "resCoordType": "WGS84GEO",
        "searchOption": "0",
    }
    if waypoints:
        if len(waypoints) > 5:
            return {'path': None, 'distance_m': 0, 'duration_ms': 0,
                    'status': f'ERROR: 경유지 {len(waypoints)}개. Tmap 한도 5개 초과'}
        payload["passList"] = "_".join(f"{wp[1]},{wp[0]}" for wp in waypoints)
    
    try:
        r = requests.post(TMAP_PEDESTRIAN_URL, headers=headers, json=payload, timeout=15)
    except requests.exceptions.RequestException as e:
        return {'path': None, 'distance_m': 0, 'duration_ms': 0,
                'status': f'ERROR: {type(e).__name__}'}
    
    if r.status_code != 200:
        return {'path': None, 'distance_m': 0, 'duration_ms': 0,
                'status': f'ERROR: HTTP {r.status_code}, body: {r.text[:150]}'}
    
    try:
        result = r.json()
    except json.JSONDecodeError:
        return {'path': None, 'distance_m': 0, 'duration_ms': 0, 'status': 'JSON_ERROR'}
    
    if result.get('type') != 'FeatureCollection':
        return {'path': None, 'distance_m': 0, 'duration_ms': 0, 'status': 'INVALID'}
    
    features = result.get('features', [])
    if not features:
        return {'path': None, 'distance_m': 0, 'duration_ms': 0, 'status': 'NO_FEATURES'}
    
    first_props = features[0].get('properties', {})
    total_distance_m = first_props.get('totalDistance', 0)
    total_time_s = first_props.get('totalTime', 0)
    
    path_lonlat = []
    for f in features:
        geom = f.get('geometry', {})
        if geom.get('type') == 'LineString':
            path_lonlat.extend(geom.get('coordinates', []))
    
    # (lat, lon) 형식으로 변환 (본 노트북 일관성)
    path_latlon = [(pt[1], pt[0]) for pt in path_lonlat]
    
    return {
        'path': path_latlon,
        'distance_m': total_distance_m,
        'duration_ms': total_time_s * 1000,
        'status': 'OK',
    }


def check_polyline_avoid_intersection(path_latlon, avoid_union_param, 
                                        avoid_union_by_code_param):
    """polyline이 회피 영역과 교차하는지 검사.
    
    Args:
        path_latlon: [(lat, lon), ...] polyline 좌표 리스트
        avoid_union_param: shapely MultiPolygon (전체 회피 영역)
        avoid_union_by_code_param: dict {uqa_code: union} (코드별 회피 영역)
    
    Returns:
        dict: crosses, intersection_length_m, intersection_ratio, crossed_uqa_codes
    """
    empty_result = {
        'crosses': False, 'intersection_length_m': 0, 
        'intersection_ratio': 0, 'crossed_uqa_codes': [],
    }
    
    if not path_latlon or len(path_latlon) < 2:
        return empty_result
    
    # polyline → shapely LineString (lon, lat 순서)
    line = LineString([(pt[1], pt[0]) for pt in path_latlon])
    
    if not line.intersects(avoid_union_param):
        return empty_result
    
    intersection = line.intersection(avoid_union_param)
    inter_length_m = intersection.length * 111000.0
    total_length_m = line.length * 111000.0
    ratio = inter_length_m / total_length_m if total_length_m > 0 else 0
    
    # 어느 UQA 코드와 교차?
    crossed_codes = []
    for code, union in avoid_union_by_code_param.items():
        if line.intersects(union):
            crossed_codes.append(code)
    
    return {
        'crosses': True,
        'intersection_length_m': inter_length_m,
        'intersection_ratio': ratio,
        'crossed_uqa_codes': crossed_codes,
    }


# === 동작 검증 — junggu·donggu에서 원본 K=3 vs 방법 C K=3 polyline 직접 비교 ===
print("[Tmap 보행자 호출 + 교차 검사 동작 검증]")
print("=" * 80)
print("방법 C가 K=3을 다르게 선택한 2개 OD에서 polyline 통과 효과 측정\n")

for district in ['junggu', 'donggu']:
    od = OD_CANDIDATES_V2[district]
    cands = od_buffer_results[district]['candidates']
    
    print(f"[{district}]")
    
    # 원본 K=3
    sel_orig, _ = algorithm_greedy_original(cands, od['O'], od['D'])
    wp_orig = [(row['latitude'], row['longitude']) 
               for _, row in sel_orig.iterrows()]
    
    # 방법 C K=3
    sel_v2, _ = algorithm_greedy_v2(cands, od['O'], od['D'],
                                      avoid_union, avoid_union_prepared)
    wp_v2 = [(row['latitude'], row['longitude'])
             for _, row in sel_v2.iterrows()]
    
    # 원본 polyline
    r_orig = get_pedestrian_route(od['O'], od['D'], waypoints=wp_orig)
    if r_orig['status'] == 'OK':
        c_orig = check_polyline_avoid_intersection(
            r_orig['path'], avoid_union, avoid_union_by_code)
        print(f"  원본 K=3 도보 polyline:")
        print(f"    거리 {r_orig['distance_m']/1000:.2f}km, 점 {len(r_orig['path'])}개")
        if c_orig['crosses']:
            print(f"    교차 {c_orig['intersection_length_m']:.0f}m "
                  f"({c_orig['intersection_ratio']*100:.2f}%), "
                  f"UQA {c_orig['crossed_uqa_codes']}")
        else:
            print(f"    교차 없음")
    else:
        print(f"  원본 polyline 호출 실패: {r_orig['status']}")
        continue
    
    # 방법 C polyline
    r_v2 = get_pedestrian_route(od['O'], od['D'], waypoints=wp_v2)
    if r_v2['status'] == 'OK':
        c_v2 = check_polyline_avoid_intersection(
            r_v2['path'], avoid_union, avoid_union_by_code)
        print(f"  방법 C K=3 도보 polyline:")
        print(f"    거리 {r_v2['distance_m']/1000:.2f}km, 점 {len(r_v2['path'])}개")
        if c_v2['crosses']:
            print(f"    교차 {c_v2['intersection_length_m']:.0f}m "
                  f"({c_v2['intersection_ratio']*100:.2f}%), "
                  f"UQA {c_v2['crossed_uqa_codes']}")
        else:
            print(f"    교차 없음")
        
        # 비교
        diff_m = c_v2['intersection_length_m'] - c_orig['intersection_length_m']
        if diff_m < 0:
            print(f"  방법 C 효과: 교차 {abs(diff_m):.0f}m 감소")
        elif diff_m > 0:
            print(f"  방법 C 효과: 교차 {diff_m:.0f}m 증가 (예상 외)")
        else:
            print(f"  방법 C 효과: 변화 없음")
    print()

In [ ]:
# 셀 10: swap 함수 — 방법 C K=3의 polyline이 통과 시 후보 교체

MAX_SWAP_ATTEMPTS = 5   # swap 최대 시도 횟수

def attempt_swap(scored_pool, initial_selection, origin, destination,
                  avoid_union_param, avoid_union_by_code_param,
                  max_attempts=MAX_SWAP_ATTEMPTS, verbose=False):
    """K=3 swap — polyline 통과 시 회피 페널티 가장 큰 핫스팟을 다음 후보로 교체.
    
    전략:
    1. 현재 K=3로 polyline 호출 + 교차 검사
    2. 교차 없으면 종료 (성공)
    3. 교차 시:
       a. 현재 K=3 중 avoid_penalty_i 가장 큰 후보 식별
       b. pool에서 K=3에 포함 안 된 다음 Score_i 순위 후보로 교체
       c. 새 K=3로 polyline 재호출
    4. max_attempts 초과 시 종료 (hard case)
    
    Args:
        scored_pool: compute_scores_with_avoidance 적용된 Top-15 풀
        initial_selection: 방법 C가 선정한 초기 K=3 DataFrame
        origin, destination: OD 좌표
        max_attempts: swap 시도 한도
    
    Returns:
        dict: final_selection, final_polyline, final_intersection, 
              final_distance_m, final_duration_ms, n_swaps, hard_case, swap_history
    """
    selected_ids = set(initial_selection['hotspot_id'])
    
    # 대체 후보: pool 중 초기 K=3에 안 들어간 후보를 Score_i 내림차순
    alternates = scored_pool[~scored_pool['hotspot_id'].isin(selected_ids)]
    alternates = alternates.sort_values('Score_i', ascending=False).reset_index(drop=True)
    
    current = initial_selection.copy().sort_values('Score_i', ascending=False).reset_index(drop=True)
    swap_history = []
    
    for attempt in range(max_attempts + 1):
        # 현재 K=3로 polyline 호출
        waypoints = [(row['latitude'], row['longitude']) 
                     for _, row in current.iterrows()]
        route = get_pedestrian_route(origin, destination, waypoints=waypoints)
        
        if route['status'] != 'OK':
            return {
                'final_selection': current, 'final_polyline': None,
                'final_intersection': None, 'final_distance_m': 0,
                'final_duration_ms': 0, 'n_swaps': attempt,
                'hard_case': True, 'swap_history': swap_history,
                'error': route['status'],
            }
        
        intersection = check_polyline_avoid_intersection(
            route['path'], avoid_union_param, avoid_union_by_code_param)
        
        # 교차 없으면 성공
        if not intersection['crosses']:
            if verbose:
                print(f"  swap {attempt}회 후 교차 해소")
            return {
                'final_selection': current,
                'final_polyline': route['path'],
                'final_intersection': intersection,
                'final_distance_m': route['distance_m'],
                'final_duration_ms': route['duration_ms'],
                'n_swaps': attempt,
                'hard_case': False,
                'swap_history': swap_history,
            }
        
        # 교차 발생 → swap 시도
        if attempt >= max_attempts or len(alternates) == 0:
            # 더 이상 swap 못 함 → hard case
            if verbose:
                print(f"  swap {attempt}회 후 hard case (한도 도달 또는 대체 후보 소진)")
            return {
                'final_selection': current,
                'final_polyline': route['path'],
                'final_intersection': intersection,
                'final_distance_m': route['distance_m'],
                'final_duration_ms': route['duration_ms'],
                'n_swaps': attempt,
                'hard_case': True,
                'swap_history': swap_history,
            }
        
        # 현재 K=3 중 페널티 가장 큰 후보 식별
        worst_idx = current['avoid_penalty_i'].idxmax()
        worst_id = current.loc[worst_idx, 'hotspot_id']
        worst_penalty = current.loc[worst_idx, 'avoid_penalty_i']
        
        # 다음 후보로 교체
        new_candidate = alternates.iloc[0]
        alternates = alternates.iloc[1:].reset_index(drop=True)
        
        if verbose:
            print(f"  swap #{attempt+1}: {worst_id} (penalty {worst_penalty:.2f}) "
                  f"→ {new_candidate['hotspot_id']}")
        
        swap_history.append({
            'attempt': attempt + 1,
            'removed': worst_id,
            'removed_penalty': worst_penalty,
            'added': new_candidate['hotspot_id'],
            'added_penalty': new_candidate['avoid_penalty_i'],
            'intersection_before_m': intersection['intersection_length_m'],
        })
        
        # K=3 갱신
        current = current.drop(index=worst_idx).reset_index(drop=True)
        new_row = new_candidate.to_frame().T
        current = pd.concat([current, new_row], ignore_index=True)
        current = current.sort_values('Score_i', ascending=False).reset_index(drop=True)
    
    # 도달 안 함
    return None


# === 동작 검증 — donggu에서 단독 swap 호출 ===
print("=" * 80)
print("[swap 함수 단독 동작 검증 — donggu]")
print("=" * 80)

district = 'donggu'
od = OD_CANDIDATES_V2[district]
cands = od_buffer_results[district]['candidates']

# 방법 C로 초기 K=3 선정
pool = cands.nlargest(min(BF_TOP_N, len(cands)), 'S_i')
scored_pool = compute_scores_with_avoidance(
    pool, od['O'], od['D'], avoid_union, avoid_union_prepared)
initial_sel = scored_pool.nlargest(K, 'Score_i')

print(f"  Buffer 후보: {len(cands)}곳 / S_i Top-{BF_TOP_N} 풀")
print(f"  초기 K=3 (방법 C 적용):")
for _, row in initial_sel.iterrows():
    print(f"    {row['hotspot_id']}: S_i={row['S_i']:.2f}, "
          f"D_i={row['D_i']:.2f}, penalty={row['avoid_penalty_i']:.2f}")

# swap 실행
print(f"\n  swap 시도 (verbose):")
swap_result = attempt_swap(
    scored_pool, initial_sel, od['O'], od['D'],
    avoid_union, avoid_union_by_code,
    verbose=True,
)

print(f"\n  최종 결과:")
print(f"    swap 횟수: {swap_result['n_swaps']}")
print(f"    hard case: {swap_result['hard_case']}")
if swap_result['final_intersection']:
    fi = swap_result['final_intersection']
    print(f"    최종 polyline 교차: {fi['intersection_length_m']:.0f}m "
          f"({fi['intersection_ratio']*100:.2f}%)")
print(f"    최종 도보 거리: {swap_result['final_distance_m']/1000:.2f}km")
print(f"    최종 K=3:")
for _, row in swap_result['final_selection'].iterrows():
    print(f"      {row['hotspot_id']}: S_i={row['S_i']:.2f}, "
          f"penalty={row['avoid_penalty_i']:.2f}")

In [ ]:
# # 셀 11: 방법 C+swap 통합 알고리즘 함수 정의
# # visit_order는 Score_i 내림차순 고정 (evaluate_path 미사용)

# def algorithm_greedy_with_avoidance(candidates_df, origin, destination,
#                                       avoid_union_param, avoid_union_prepared_param,
#                                       avoid_union_by_code_param,
#                                       alpha=ALPHA, beta=BETA, gamma=GAMMA,
#                                       k=K, max_swap=MAX_SWAP_ATTEMPTS,
#                                       verbose=False):
#     """방법 C + swap 통합 알고리즘 (4번째 알고리즘 후보 E_v2).
    
#     파이프라인:
#     1. Buffer 후보 → S_i Top-15 풀
#     2. 방법 C: Score_i = α·S_i - β·D_i - γ·avoid_penalty_i
#     3. Score_i Top-K=3 초기 선정
#     4. K=3을 Score_i 내림차순으로 방문 (visit_order 최적화 없음)
#     5. Tmap polyline 호출 + 교차 검사
#     6. 교차 시 swap (penalty 큰 후보를 다음 Score 후보로 교체)
#     7. 최대 max_swap회까지 반복 후 hard case 종료
    
#     Args:
#         candidates_df: Buffer 후보 DataFrame
#         origin, destination: (lat, lon)
#         avoid_*: 셀 3에서 만든 회피 영역 객체들
#         gamma: 회피 페널티 가중치 (기본 1.0)
#         max_swap: swap 최대 시도 (기본 5)
#         verbose: 진행 출력 여부
    
#     Returns:
#         dict: final_selection, visit_order, final_polyline, final_intersection,
#               final_distance_m, final_duration_ms, n_swaps, hard_case,
#               swap_history, total_elapsed_ms
#     """
#     t0 = time.time()
    
#     # 1. S_i Top-15 풀
#     n = len(candidates_df)
#     if n == 0:
#         return None
#     pool = candidates_df.nlargest(min(BF_TOP_N, n), 'S_i')
    
#     # 2. 방법 C: Score_i 계산
#     scored_pool = compute_scores_with_avoidance(
#         pool, origin, destination,
#         avoid_union_param, avoid_union_prepared_param,
#         alpha=alpha, beta=beta, gamma=gamma,
#     )
    
#     # 3. Score_i Top-K=3 초기 선정
#     k_eff = min(k, n)
#     initial_selection = scored_pool.nlargest(k_eff, 'Score_i')
    
#     # 대체 후보: pool 중 K=3에 없는 것, Score_i 내림차순
#     alternates = scored_pool[~scored_pool['hotspot_id'].isin(initial_selection['hotspot_id'])]
#     alternates = alternates.sort_values('Score_i', ascending=False).reset_index(drop=True)
    
#     # 현재 K=3 (Score_i 내림차순으로 정렬)
#     current = initial_selection.copy().sort_values('Score_i', ascending=False).reset_index(drop=True)
#     swap_history = []
    
#     if verbose:
#         print(f"  초기 K=3 선정 완료. swap 루프 진입.")
    
#     # 4·5·6. polyline + swap 루프 (visit_order = Score_i 순서 고정)
#     for attempt in range(max_swap + 1):
#         # 4. K=3을 Score_i 내림차순(현재 정렬 상태)으로 방문
#         visit_order = current['hotspot_id'].tolist()
#         ordered_waypoints = [(row['latitude'], row['longitude'])
#                               for _, row in current.iterrows()]
        
#         # 5. Tmap polyline 호출
#         route = get_pedestrian_route(origin, destination, waypoints=ordered_waypoints)
#         if route['status'] != 'OK':
#             return {
#                 'final_selection': current, 'visit_order': visit_order,
#                 'final_polyline': None, 'final_intersection': None,
#                 'final_distance_m': 0, 'final_duration_ms': 0,
#                 'n_swaps': attempt, 'hard_case': True,
#                 'swap_history': swap_history,
#                 'total_elapsed_ms': (time.time() - t0) * 1000,
#                 'error': route['status'],
#             }
        
#         intersection = check_polyline_avoid_intersection(
#             route['path'], avoid_union_param, avoid_union_by_code_param)
        
#         if verbose:
#             cross_info = (f"{intersection['intersection_length_m']:.0f}m"
#                           if intersection['crosses'] else "없음")
#             print(f"  attempt {attempt}: 교차 {cross_info}")
        
#         # 6a. 교차 없으면 성공 종료
#         if not intersection['crosses']:
#             return {
#                 'final_selection': current,
#                 'visit_order': visit_order,
#                 'final_polyline': route['path'],
#                 'final_intersection': intersection,
#                 'final_distance_m': route['distance_m'],
#                 'final_duration_ms': route['duration_ms'],
#                 'n_swaps': attempt,
#                 'hard_case': False,
#                 'swap_history': swap_history,
#                 'total_elapsed_ms': (time.time() - t0) * 1000,
#             }
        
#         # 6b. swap 한도 도달 → hard case 종료
#         if attempt >= max_swap or len(alternates) == 0:
#             return {
#                 'final_selection': current,
#                 'visit_order': visit_order,
#                 'final_polyline': route['path'],
#                 'final_intersection': intersection,
#                 'final_distance_m': route['distance_m'],
#                 'final_duration_ms': route['duration_ms'],
#                 'n_swaps': attempt,
#                 'hard_case': True,
#                 'swap_history': swap_history,
#                 'total_elapsed_ms': (time.time() - t0) * 1000,
#             }
        
#         # 6c. swap 시도: penalty 가장 큰 후보 → 다음 Score 후보
#         worst_idx = current['avoid_penalty_i'].idxmax()
#         worst_id = current.loc[worst_idx, 'hotspot_id']
#         worst_penalty = current.loc[worst_idx, 'avoid_penalty_i']
        
#         new_candidate = alternates.iloc[0]
#         alternates = alternates.iloc[1:].reset_index(drop=True)
        
#         swap_history.append({
#             'attempt': attempt + 1,
#             'removed': worst_id,
#             'removed_penalty': worst_penalty,
#             'added': new_candidate['hotspot_id'],
#             'added_penalty': new_candidate['avoid_penalty_i'],
#             'intersection_before_m': intersection['intersection_length_m'],
#         })
        
#         # K=3 갱신 (Score_i 내림차순 재정렬)
#         current = current.drop(index=worst_idx).reset_index(drop=True)
#         new_row = new_candidate.to_frame().T
#         current = pd.concat([current, new_row], ignore_index=True)
#         current = current.sort_values('Score_i', ascending=False).reset_index(drop=True)
    
#     return None   # 도달 안 함


# print("[통합 알고리즘 함수 정의 완료]")
# print(f"  파이프라인: Buffer → S_i Top-{BF_TOP_N} → 방법 C Score_i Top-{K} → Tmap → swap")
# print(f"  visit_order: Score_i 내림차순 (최적화 없음)")
# print(f"  하이퍼파라미터: α={ALPHA}, β={BETA}, γ={GAMMA}, K={K}, max_swap={MAX_SWAP_ATTEMPTS}")


# # === 단독 동작 검증 — junggu ===
# print(f"\n{'=' * 80}")
# print(f"[통합 알고리즘 단독 검증 — junggu]")
# print(f"{'=' * 80}")

# district = 'junggu'
# od = OD_CANDIDATES_V2[district]
# cands = od_buffer_results[district]['candidates']

# result = algorithm_greedy_with_avoidance(
#     cands, od['O'], od['D'],
#     avoid_union, avoid_union_prepared, avoid_union_by_code,
#     verbose=True,
# )

# print(f"\n  실행 시간: {result['total_elapsed_ms']:.1f}ms")
# print(f"  swap 횟수: {result['n_swaps']}")
# print(f"  hard case: {result['hard_case']}")

# print(f"\n  최종 K=3 (방문 순서):")
# sel = result['final_selection']
# for i, hid in enumerate(result['visit_order'], 1):
#     row = sel[sel['hotspot_id'] == hid].iloc[0]
#     print(f"    {i}. {hid}: S_i={row['S_i']:.2f}, "
#           f"D_i={row['D_i']:.2f}, penalty={row['avoid_penalty_i']:.2f}, "
#           f"Score_i={row['Score_i']:.3f}")

# print(f"\n  최종 polyline:")
# print(f"    도보 거리: {result['final_distance_m']/1000:.2f}km")
# print(f"    도보 소요: {result['final_duration_ms']/60000:.1f}분")
# print(f"    점 수: {len(result['final_polyline'])}개")

# fi = result['final_intersection']
# if fi and fi['crosses']:
#     print(f"  회피 영역 교차: {fi['intersection_length_m']:.0f}m "
#           f"({fi['intersection_ratio']*100:.2f}%)")
#     print(f"  UQA 코드: {fi['crossed_uqa_codes']}")
# else:
#     print(f"  회피 영역 교차: 없음")

# if result['swap_history']:
#     print(f"\n  swap 이력:")
#     for h in result['swap_history']:
#         print(f"    #{h['attempt']}: {h['removed']} (p={h['removed_penalty']:.2f}) "
#               f"→ {h['added']}")

In [ ]:
# 셀 11 (수정): 방법 C+swap 통합 알고리즘 + 최선 결과 추적 안전장치

def algorithm_greedy_with_avoidance(candidates_df, origin, destination,
                                      avoid_union_param, avoid_union_prepared_param,
                                      avoid_union_by_code_param,
                                      alpha=ALPHA, beta=BETA, gamma=GAMMA,
                                      k=K, max_swap=MAX_SWAP_ATTEMPTS,
                                      verbose=False):
    """방법 C + swap 통합 알고리즘 (4번째 알고리즘 후보 E_v2).
    
    파이프라인:
    1. Buffer 후보 → S_i Top-15 풀
    2. 방법 C: Score_i = α·S_i - β·D_i - γ·avoid_penalty_i
    3. Score_i Top-K=3 초기 선정
    4. K=3을 Score_i 내림차순으로 방문
    5. Tmap polyline 호출 + 교차 검사
    6. 교차 시 swap (penalty 큰 후보 → 다음 Score 후보)
    7. ★ 매 attempt마다 best 추적 → 최종적으로 best 반환
    8. 최대 max_swap회까지 반복 후 hard case 종료
    
    수정 (단계 6-α 발견 반영):
    - 최선 결과 추적 (best dict)
    - swap이 결과를 악화시켜도 baseline(또는 더 좋은) 결과 보장
    - 'best_from_attempt' 필드 추가 — 어느 attempt가 best였는지 명시
    
    Returns:
        dict: final_selection, visit_order, final_polyline, final_intersection,
              final_distance_m, final_duration_ms, n_swaps, hard_case,
              swap_history, total_elapsed_ms, best_from_attempt
    """
    t0 = time.time()
    
    # 1. S_i Top-15 풀
    n = len(candidates_df)
    if n == 0:
        return None
    pool = candidates_df.nlargest(min(BF_TOP_N, n), 'S_i')
    
    # 2. 방법 C: Score_i 계산
    scored_pool = compute_scores_with_avoidance(
        pool, origin, destination,
        avoid_union_param, avoid_union_prepared_param,
        alpha=alpha, beta=beta, gamma=gamma,
    )
    
    # 3. Score_i Top-K=3 초기 선정
    k_eff = min(k, n)
    initial_selection = scored_pool.nlargest(k_eff, 'Score_i')
    
    # 대체 후보: pool 중 K=3에 없는 것, Score_i 내림차순
    alternates = scored_pool[~scored_pool['hotspot_id'].isin(initial_selection['hotspot_id'])]
    alternates = alternates.sort_values('Score_i', ascending=False).reset_index(drop=True)
    
    current = initial_selection.copy().sort_values('Score_i', ascending=False).reset_index(drop=True)
    swap_history = []
    
    # ★ 최선 결과 추적 (NEW)
    best = {
        'cross_m': None,
        'selection': None,
        'visit_order': None,
        'polyline': None,
        'intersection': None,
        'distance_m': 0,
        'duration_ms': 0,
        'attempt_no': None,
    }
    
    if verbose:
        print(f"  초기 K=3 선정 완료. swap 루프 진입.")
    
    for attempt in range(max_swap + 1):
        # 4. K=3 방문 순서 = Score_i 내림차순
        visit_order = current['hotspot_id'].tolist()
        ordered_waypoints = [(row['latitude'], row['longitude'])
                              for _, row in current.iterrows()]
        
        # 5. Tmap polyline 호출
        route = get_pedestrian_route(origin, destination, waypoints=ordered_waypoints)
        if route['status'] != 'OK':
            # API 에러 → best가 있으면 best 반환
            if best['selection'] is not None:
                return {
                    'final_selection': best['selection'],
                    'visit_order': best['visit_order'],
                    'final_polyline': best['polyline'],
                    'final_intersection': best['intersection'],
                    'final_distance_m': best['distance_m'],
                    'final_duration_ms': best['duration_ms'],
                    'n_swaps': attempt,
                    'hard_case': True,
                    'swap_history': swap_history,
                    'total_elapsed_ms': (time.time() - t0) * 1000,
                    'best_from_attempt': best['attempt_no'],
                    'error': route['status'],
                }
            return {
                'final_selection': current, 'visit_order': visit_order,
                'final_polyline': None, 'final_intersection': None,
                'final_distance_m': 0, 'final_duration_ms': 0,
                'n_swaps': attempt, 'hard_case': True,
                'swap_history': swap_history,
                'total_elapsed_ms': (time.time() - t0) * 1000,
                'error': route['status'],
            }
        
        intersection = check_polyline_avoid_intersection(
            route['path'], avoid_union_param, avoid_union_by_code_param)
        current_cross_m = (intersection['intersection_length_m'] 
                           if intersection['crosses'] else 0)
        
        # ★ best 갱신 (NEW) — 교차 길이가 작을수록 좋음
        if best['cross_m'] is None or current_cross_m < best['cross_m']:
            best['cross_m'] = current_cross_m
            best['selection'] = current.copy()
            best['visit_order'] = visit_order
            best['polyline'] = route['path']
            best['intersection'] = intersection
            best['distance_m'] = route['distance_m']
            best['duration_ms'] = route['duration_ms']
            best['attempt_no'] = attempt
        
        if verbose:
            cross_info = (f"{intersection['intersection_length_m']:.0f}m"
                          if intersection['crosses'] else "없음")
            is_best = " ★best 갱신" if best['attempt_no'] == attempt else ""
            print(f"  attempt {attempt}: 교차 {cross_info}{is_best}")
        
        # 6a. 교차 없으면 성공 종료 (즉시 종료, best도 자동으로 현재)
        if not intersection['crosses']:
            return {
                'final_selection': current,
                'visit_order': visit_order,
                'final_polyline': route['path'],
                'final_intersection': intersection,
                'final_distance_m': route['distance_m'],
                'final_duration_ms': route['duration_ms'],
                'n_swaps': attempt,
                'hard_case': False,
                'swap_history': swap_history,
                'total_elapsed_ms': (time.time() - t0) * 1000,
                'best_from_attempt': attempt,
            }
        
        # 6b. swap 한도 도달 → ★ best 반환 (NEW)
        if attempt >= max_swap or len(alternates) == 0:
            return {
                'final_selection': best['selection'],
                'visit_order': best['visit_order'],
                'final_polyline': best['polyline'],
                'final_intersection': best['intersection'],
                'final_distance_m': best['distance_m'],
                'final_duration_ms': best['duration_ms'],
                'n_swaps': attempt,
                'hard_case': True,
                'swap_history': swap_history,
                'total_elapsed_ms': (time.time() - t0) * 1000,
                'best_from_attempt': best['attempt_no'],
            }
        
        # 6c. swap 시도
        worst_idx = current['avoid_penalty_i'].idxmax()
        worst_id = current.loc[worst_idx, 'hotspot_id']
        worst_penalty = current.loc[worst_idx, 'avoid_penalty_i']
        
        new_candidate = alternates.iloc[0]
        alternates = alternates.iloc[1:].reset_index(drop=True)
        
        swap_history.append({
            'attempt': attempt + 1,
            'removed': worst_id,
            'removed_penalty': worst_penalty,
            'added': new_candidate['hotspot_id'],
            'added_penalty': new_candidate['avoid_penalty_i'],
            'intersection_before_m': current_cross_m,
        })
        
        # K=3 갱신
        current = current.drop(index=worst_idx).reset_index(drop=True)
        new_row = new_candidate.to_frame().T
        current = pd.concat([current, new_row], ignore_index=True)
        current = current.sort_values('Score_i', ascending=False).reset_index(drop=True)
    
    return None   # 도달 안 함


print("[통합 알고리즘 함수 정의 완료 — 안전장치 추가]")
print(f"  파이프라인: Buffer → S_i Top-{BF_TOP_N} → 방법 C Score_i Top-{K} → Tmap → swap")
print(f"  ★ NEW: 최선 결과 추적. swap이 결과를 악화시킬 수 없음.")
print(f"  하이퍼파라미터: α={ALPHA}, β={BETA}, γ={GAMMA}, K={K}, max_swap={MAX_SWAP_ATTEMPTS}")


# === 단독 동작 검증 — junggu (best 추적 확인) ===
print(f"\n{'=' * 80}")
print(f"[통합 알고리즘 단독 검증 — junggu (best 추적 확인)]")
print(f"{'=' * 80}")

district = 'junggu'
od = OD_CANDIDATES_V2[district]
cands = od_buffer_results[district]['candidates']

result = algorithm_greedy_with_avoidance(
    cands, od['O'], od['D'],
    avoid_union, avoid_union_prepared, avoid_union_by_code,
    verbose=True,
)

print(f"\n  실행 시간: {result['total_elapsed_ms']:.1f}ms")
print(f"  swap 횟수: {result['n_swaps']}")
print(f"  hard case: {result['hard_case']}")
print(f"  ★ best 채택 시점: attempt {result['best_from_attempt']}")

print(f"\n  최종 K=3 (best 시점):")
sel = result['final_selection']
for i, hid in enumerate(result['visit_order'], 1):
    row = sel[sel['hotspot_id'] == hid].iloc[0]
    print(f"    {i}. {hid}: S_i={row['S_i']:.2f}, "
          f"D_i={row['D_i']:.2f}, penalty={row['avoid_penalty_i']:.2f}")

print(f"\n  최종 polyline:")
print(f"    도보 거리: {result['final_distance_m']/1000:.2f}km")
print(f"    회피 교차: {result['final_intersection']['intersection_length_m']:.0f}m"
      if result['final_intersection']['crosses'] else "    회피 교차: 없음")

---

## 단계 6-δ: K 스윕 (v5 신규)

여기부터 v5 신규 셀이다. 선행 조건은 위 셀 1~14가 실행되어 `hotspots`, `avoid_union`,
`algorithm_greedy_with_avoidance` 등이 정의된 상태이며, `output4_260731/` 폴더에
v4 산출물(`realistic_od_200.pkl`, `step6_gamma_200od_realistic.csv`)이 존재해야 한다.


In [ ]:
# ==========================================================================
# 셀 15: OD 표본 로드 (v4_260731 표본 재사용)
# ==========================================================================
# K 비교의 전제는 동일 표본이다. v4(260731)에서 생성·검증한 현실적 인접 OD
# 200쌍을 그대로 사용하며, 신규 생성하지 않는다. 표본이 달라지면 K=3 결과
# (v4에서 이미 측정 완료)와 비교할 수 없다.
#
# OD 생성은 Tmap을 호출하지 않으므로 재생성해도 API 비용은 없으나,
# 시드가 같아도 폴리곤 선택 난수 흐름이 달라질 수 있어 로드 방식을 택한다.

import pickle
from pathlib import Path

# === 경로 설정 ===
V4_DIR = Path("./output4_260731")          # v4 산출물 (읽기 전용)
V5_OUTPUT_DIR = Path("./output5_260821")   # v5 전용 출력
V5_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OD_PKL   = V4_DIR / "realistic_od_200.pkl"
K3_CSV   = V4_DIR / "step6_gamma_200od_realistic.csv"   # K=3 기존 결과

print(f"[v4 참조 폴더] {V4_DIR.resolve()}")
print(f"[v5 출력 폴더] {V5_OUTPUT_DIR.resolve()}")

if not OD_PKL.exists():
    raise FileNotFoundError(
        f"{OD_PKL} 를 찾을 수 없다. v4_260731 노트북을 먼저 실행하여 OD를 생성해야 한다.")

with open(OD_PKL, "rb") as f:
    REALISTIC_OD = pickle.load(f)

print(f"\n[OD 로드] {OD_PKL.name} — {len(REALISTIC_OD)}쌍")

# === 표본 무결성 확인 (v4와 동일 표본인지) ===
df_od5 = pd.DataFrame([{k: v for k, v in od.items() if k != 'candidates'}
                       for od in REALISTIC_OD])
print(f"  직선 거리(km): 평균 {df_od5['direct_km'].mean():.2f}")
print(f"  직선의 회피영역 통과 길이(m): 중앙값 {df_od5['chord_m'].median():.0f}")
print(f"  Buffer 후보 수: 평균 {df_od5['n_candidates'].mean():.1f} "
      f"[{df_od5['n_candidates'].min()}, {df_od5['n_candidates'].max()}]")
print(f"  군구 수: {df_od5['district'].nunique()}개")

# === K 상향 시 후보 부족 사전 점검 ===
# k_eff = min(k, n) 이므로 후보가 K보다 적으면 요청한 개수를 채우지 못한다.
print("\n[후보 부족 사전 점검]")
for k in [3, 4, 5]:
    short = int((df_od5['n_candidates'] < k).sum())
    print(f"  K={k}: 후보 부족 OD {short}쌍 ({short/len(df_od5)*100:.1f}%)")

# === K=3 기존 결과 존재 확인 ===
if K3_CSV.exists():
    _k3 = pd.read_csv(K3_CSV)
    print(f"\n[K=3 기존 결과] {K3_CSV.name} — {len(_k3)}행 (v4_260731에서 측정 완료)")
    print("  셀 17 집계 시 비교군으로 사용한다. 재실행 불필요.")
else:
    print(f"\n[경고] {K3_CSV} 없음. 셀 17에서 K=3 비교가 생략된다.")


### 셀 16 실행 전 확인

`RUN_K` 값을 확인한다. **1일차는 4, 2일차는 5**로 설정한다.
실행 후 셀 하단의 `[다음 단계 안내]`가 다음에 할 일을 알려준다.


In [ ]:
# ==========================================================================
# 셀 16: K 스윕 실행 (RUN_K 값을 바꿔가며 분할 실행)
# ==========================================================================
# ┌──────────────────────────────────────────────────────────────────────┐
# │ 실행 방법                                                             │
# │   1일차: RUN_K = 4 로 두고 이 셀 실행 (약 900회 호출)                 │
# │   2일차: RUN_K = 5 로 바꾸고 이 셀 실행 (약 900회 호출)               │
# │                                                                       │
# │ checkpoint가 K별로 분리 저장되므로 두 실행이 서로 덮어쓰지 않는다.     │
# │ 중단되어도 완료된 배치는 보존되며, 재실행 시 건너뛴다.                 │
# └──────────────────────────────────────────────────────────────────────┘

RUN_K = 4            # <<< 1일차: 4 / 2일차: 5 로 변경 후 실행

GAMMA_FIXED = 1.0    # v3에서 확정. K 비교를 위해 고정한다.
COORD_ROUND = 6
BATCH_SIZE  = 25     # 200 OD -> 8배치
MAX_BATCHES_PER_RUN = 8   # 1회 실행 시 처리할 최대 배치 수 (한도 관리용)
FORCE_RERUN = False  # True로 두면 현재 RUN_K의 checkpoint를 삭제하고 재실행

# === Tmap 응답 캐시 (v3·v4와 동일 구조) ===
if not getattr(get_pedestrian_route, '_is_cache_wrapper', False):
    _get_pedestrian_route_raw = get_pedestrian_route
    _tmap_cache = {}
    _tmap_stats = {'hit': 0, 'miss': 0}

    def get_pedestrian_route(origin, destination, waypoints=None, debug=False):
        key = (
            round(origin[0], COORD_ROUND), round(origin[1], COORD_ROUND),
            round(destination[0], COORD_ROUND), round(destination[1], COORD_ROUND),
            tuple((round(w[0], COORD_ROUND), round(w[1], COORD_ROUND))
                  for w in (waypoints or [])),
        )
        if key in _tmap_cache:
            _tmap_stats['hit'] += 1
            return _tmap_cache[key]
        _tmap_stats['miss'] += 1
        res = _get_pedestrian_route_raw(origin, destination, waypoints=waypoints, debug=debug)
        if res.get('status') == 'OK':
            _tmap_cache[key] = res
        return res

    get_pedestrian_route._is_cache_wrapper = True
    print("[Tmap 응답 캐시 적용] get_pedestrian_route 래핑 완료")
else:
    _tmap_cache.clear()
    _tmap_stats['hit'] = 0
    _tmap_stats['miss'] = 0
    print("[Tmap 응답 캐시 초기화] 기존 래퍼 재사용")


def _run_one_od_k(od, k):
    """OD 1개를 지정 K로 실행한다. 원본(동일 K)과 방법 C+swap을 함께 측정한다."""
    # 원본 알고리즘 (회피 페널티 없음), 동일 K 적용
    sel_o, _ = algorithm_greedy_original(od['candidates'], od['O'], od['D'], k=k)
    if len(sel_o) == 0:
        return None
    wp = [(r['latitude'], r['longitude']) for _, r in sel_o.iterrows()]
    route_o = get_pedestrian_route(od['O'], od['D'], waypoints=wp)
    if route_o['status'] != 'OK':
        return None
    inter_o = check_polyline_avoid_intersection(route_o['path'], avoid_union, avoid_union_by_code)
    orig_cross_m = inter_o['intersection_length_m'] if inter_o['crosses'] else 0.0
    orig_dist_km = route_o['distance_m'] / 1000

    # 방법 C + swap, 동일 K 적용
    res = algorithm_greedy_with_avoidance(
        od['candidates'], od['O'], od['D'],
        avoid_union, avoid_union_prepared, avoid_union_by_code,
        gamma=GAMMA_FIXED, k=k,
    )
    if res is None or res.get('final_polyline') is None:
        return None
    fi = res['final_intersection']
    v2_cross_m = fi['intersection_length_m'] if (fi and fi['crosses']) else 0.0
    v2_dist_km = res['final_distance_m'] / 1000

    return {
        'od_id': od['od_id'], 'district': od['district'],
        'margin_target_m': od['margin_target_m'], 'od_margin_m': od['od_margin_m'],
        'direct_km': od['direct_km'], 'chord_m': od['chord_m'],
        'n_candidates': od['n_candidates'], 'n_avoid_in_buffer': od['n_avoid_in_buffer'],
        'gamma': GAMMA_FIXED,
        'k_requested': k,
        'k_effective': len(res['final_selection']),   # 후보 부족 시 요청보다 작을 수 있다
        'orig_cross_m': orig_cross_m, 'v2_cross_m': v2_cross_m,
        'diff_m': v2_cross_m - orig_cross_m,
        'orig_dist_km': orig_dist_km, 'v2_dist_km': v2_dist_km,
        'dist_delta_km': v2_dist_km - orig_dist_km,
        'orig_cross_ratio_pct': orig_cross_m / (orig_dist_km * 1000) * 100 if orig_dist_km > 0 else 0.0,
        'v2_cross_ratio_pct': v2_cross_m / (v2_dist_km * 1000) * 100 if v2_dist_km > 0 else 0.0,
        'n_swaps': res['n_swaps'], 'hard_case': bool(res['hard_case']),
        'best_from_attempt': res.get('best_from_attempt'),
    }


n_batches = (len(REALISTIC_OD) + BATCH_SIZE - 1) // BATCH_SIZE

print("=" * 92)
print(f"[단계 6-delta K 스윕 — K={RUN_K} / {len(REALISTIC_OD)} OD / "
      f"{n_batches}배치 / γ={GAMMA_FIXED} 고정]")
print("=" * 92)

if FORCE_RERUN:
    for b in range(n_batches):
        ck = V5_OUTPUT_DIR / f"step6_k{RUN_K}_batch{b}.csv"
        if ck.exists():
            ck.unlink()
    print(f"[FORCE_RERUN] K={RUN_K} checkpoint 전체 삭제")

t_start = time.time()
n_run_this_session = 0

for b in range(n_batches):
    ckpt = V5_OUTPUT_DIR / f"step6_k{RUN_K}_batch{b}.csv"
    if ckpt.exists():
        print(f"  배치 {b} 건너뜀 (checkpoint 존재: {ckpt.name})")
        continue

    if n_run_this_session >= MAX_BATCHES_PER_RUN:
        print(f"  [중단] 이번 실행 배치 상한({MAX_BATCHES_PER_RUN}) 도달. "
              f"남은 배치는 다음 실행에서 이어서 처리된다.")
        break

    batch_ods = REALISTIC_OD[b * BATCH_SIZE:(b + 1) * BATCH_SIZE]
    rows, n_fail = [], 0
    for od in batch_ods:
        r = _run_one_od_k(od, RUN_K)
        if r is None:
            n_fail += 1
            continue
        rows.append(r)

    pd.DataFrame(rows).to_csv(ckpt, index=False, encoding='utf-8-sig')
    n_run_this_session += 1
    print(f"  배치 {b} 완료: OD {len(batch_ods)}개, 성공 {len(rows)}, 실패 {n_fail} "
          f"| 경과 {time.time()-t_start:.0f}s "
          f"| Tmap 호출 {_tmap_stats['miss']} (적중 {_tmap_stats['hit']})")

# === 현재 K의 배치 병합 ===
done = [V5_OUTPUT_DIR / f"step6_k{RUN_K}_batch{b}.csv" for b in range(n_batches)]
done = [f for f in done if f.exists()]

if done:
    df_k = pd.concat([pd.read_csv(f) for f in done], ignore_index=True)
    k_csv = V5_OUTPUT_DIR / f"step6_k{RUN_K}_200od.csv"
    df_k.to_csv(k_csv, index=False, encoding='utf-8-sig')
    print(f"\n[K={RUN_K} 병합] {len(df_k)}행 ({len(done)}/{n_batches}배치) -> {k_csv.name}")
    if len(done) < n_batches:
        print(f"  [미완료] {n_batches - len(done)}배치 남음. 동일 RUN_K로 재실행하면 이어서 처리된다.")
    else:
        print(f"  [완료] K={RUN_K} 전 배치 처리 완료.")
        eff = df_k['k_effective']
        print(f"  실제 경유 핫스팟 수: 평균 {eff.mean():.2f} "
              f"(요청 {RUN_K}개 미달 {int((eff < RUN_K).sum())}쌍)")
else:
    print(f"\n[K={RUN_K}] 완료된 배치 없음.")

n_calls = _tmap_stats['hit'] + _tmap_stats['miss']
if n_calls:
    print(f"\n  이번 실행 Tmap 호출 {_tmap_stats['miss']}회 / 캐시 적중 {_tmap_stats['hit']}회")
else:
    print("\n  이번 실행 Tmap 호출 없음 (전 배치 건너뜀)")

print(f"\n{'─'*92}")
print("[다음 단계 안내]")
_have = {k: (V5_OUTPUT_DIR / f'step6_k{k}_200od.csv').exists() for k in [4, 5]}
if not _have[4]:
    print("  RUN_K=4 로 두고 이 셀을 실행한다.")
elif not _have[5]:
    print("  K=4 완료. 다음 실행에서 RUN_K=5 로 바꾸고 이 셀을 실행한다.")
    print("  (오늘은 여기서 중단해도 된다. 셀 17을 실행하면 K=3 vs K=4 중간 집계를 볼 수 있다.)")
else:
    print("  K=4·K=5 모두 완료. 셀 17~19를 실행하여 집계·시각화를 진행한다.")


In [ ]:
# ==========================================================================
# 셀 17: K별 집계 및 비교 (가용한 K만 자동 집계)
# ==========================================================================
# K=3은 v4_260731 결과를 재사용하고, K=4·K=5는 v5 산출물을 읽는다.
# 아직 실행하지 않은 K는 자동으로 제외되므로, 1일차에도 중간 집계가 가능하다.
# 감소율은 총량 기준(total_reduction_pct)을 사용한다. 개별 평균은 원본 교차가
# 짧은 OD에서 비율이 폭발하여 이상치에 지배되기 때문이다(v4 6-1절).

def _load_k(k):
    """K별 결과를 로드한다. 없으면 None."""
    if k == 3:
        p = K3_CSV
    else:
        p = V5_OUTPUT_DIR / f"step6_k{k}_200od.csv"
    if not p.exists():
        return None
    df = pd.read_csv(p)
    if 'k_requested' not in df.columns:
        df['k_requested'] = k            # v4 CSV에는 K 컬럼이 없다
    return df


def _stats_k(df):
    crossing = df[df['orig_cross_m'] > 0]
    if len(crossing) > 0:
        tot = (crossing['orig_cross_m'].sum() - crossing['v2_cross_m'].sum()) \
              / crossing['orig_cross_m'].sum() * 100
        full = int(((crossing['orig_cross_m'] - crossing['v2_cross_m'])
                    / crossing['orig_cross_m'] > 0.999).sum())
    else:
        tot, full = 0.0, 0
    resid = df[df['v2_cross_m'] > 0]['v2_cross_m']
    return {
        'n': len(df),
        'n_orig_crossing': int((df['orig_cross_m'] > 0).sum()),
        'mean_swaps': df['n_swaps'].mean(),
        'hard_case_pct': df['hard_case'].mean() * 100,
        'total_reduction_pct': tot,
        'n_full_removed': full,
        'n_worse_vs_orig': int((df['diff_m'] > 1).sum()),
        'orig_dist_km_mean': df['orig_dist_km'].mean(),
        'v2_dist_km_mean': df['v2_dist_km'].mean(),
        'mean_dist_delta_km': df['dist_delta_km'].mean(),
        'residual_median_m': resid.median() if len(resid) else 0.0,
    }


pd.set_option('display.float_format', lambda x: f"{x:.2f}")

k_frames = {}
for k in [3, 4, 5]:
    df = _load_k(k)
    if df is not None:
        k_frames[k] = df

print("=" * 100)
print(f"[단계 6-delta K 스윕 집계 — 가용 K: {sorted(k_frames)}]")
print("=" * 100)

if not k_frames:
    print("  집계 가능한 결과가 없다. 셀 16을 먼저 실행한다.")
else:
    rows = []
    for k, df in k_frames.items():
        s = _stats_k(df); s['K'] = k
        rows.append(s)
    df_ksum = pd.DataFrame(rows)[
        ['K', 'n', 'n_orig_crossing', 'mean_swaps', 'hard_case_pct',
         'total_reduction_pct', 'n_full_removed', 'n_worse_vs_orig',
         'orig_dist_km_mean', 'v2_dist_km_mean', 'mean_dist_delta_km',
         'residual_median_m']
    ]
    print(df_ksum.to_string(index=False))

    # === 도보 거리 차이 (UX 옵션 차별화 근거) ===
    print("\n" + "-" * 100)
    print("[UX 옵션 관점 — K별 도보 거리]")
    base = df_ksum[df_ksum['K'] == 3]['v2_dist_km_mean']
    base_v = float(base.iloc[0]) if len(base) else None
    for _, r in df_ksum.iterrows():
        delta = f"{r['v2_dist_km_mean'] - base_v:+.2f}km" if base_v is not None else "-"
        print(f"  K={int(r['K'])}: 평균 도보 {r['v2_dist_km_mean']:.2f}km (K=3 대비 {delta})")
    if base_v is not None and len(df_ksum) > 1:
        span = df_ksum['v2_dist_km_mean'].max() - df_ksum['v2_dist_km_mean'].min()
        print(f"  => K 구간 전체 거리 폭: {span:.2f}km")
        if span < 0.5:
            print("     사용자가 체감할 만한 차이가 아닐 수 있다. 옵션 차별화 방식 재검토 필요.")

    # === 회피 성능 변화 ===
    print("\n" + "-" * 100)
    print("[회피 성능 관점 — K 증가의 영향]")
    if 3 in k_frames and len(df_ksum) > 1:
        r3 = df_ksum[df_ksum['K'] == 3].iloc[0]
        for _, r in df_ksum[df_ksum['K'] != 3].iterrows():
            dh = r['hard_case_pct'] - r3['hard_case_pct']
            dr = r['total_reduction_pct'] - r3['total_reduction_pct']
            print(f"  K={int(r['K'])}: hard case {dh:+.1f}%p | 총 감소율 {dr:+.1f}%p")
        print("  (hard case가 크게 상승하면 해당 K 옵션은 회피 성능을 희생한다는 의미다)")

    # === 후보 부족 실태 ===
    print("\n" + "-" * 100)
    print("[후보 부족 실태 — 요청 K를 채우지 못한 OD]")
    for k, df in k_frames.items():
        if 'k_effective' in df.columns:
            short = int((df['k_effective'] < k).sum())
            print(f"  K={k}: {short}쌍 ({short/len(df)*100:.1f}%) — 실제 평균 {df['k_effective'].mean():.2f}개")
        else:
            print(f"  K={k}: k_effective 컬럼 없음 (v4 결과, K=3 고정 실행)")

    ksum_csv = V5_OUTPUT_DIR / 'step6_k_sweep_summary.csv'
    df_ksum.to_csv(ksum_csv, index=False, encoding='utf-8-sig')
    print(f"\n[CSV 저장] {ksum_csv.resolve()}")

pd.reset_option('display.float_format')


### 시각화

K=4·K=5가 모두 완료된 뒤 실행한다. 셀 18은 차트 4종을, 셀 19는 동일 OD에 대해
K별 경로를 색을 달리해 중첩한 지도를 생성한다(회색=K3, 파랑=K4, 주황=K5).


In [ ]:
# ==========================================================================
# 셀 18: K 스윕 시각화 — 정적 차트 4종
# ==========================================================================
# 셀 17이 만든 df_ksum / k_frames 에 의존한다.

import matplotlib
import matplotlib.pyplot as plt

try:
    plt.rcParams['font.family'] = 'Malgun Gothic'   # Windows 기준
except Exception:
    pass
plt.rcParams['axes.unicode_minus'] = False

if not k_frames:
    print("[차트 생략] 집계 결과가 없다.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    ks = sorted(k_frames)
    x = np.arange(len(ks))

    # (1) K별 hard case 비율 / 총 감소율
    ax = axes[0, 0]
    w = 0.35
    hc = [float(df_ksum[df_ksum['K'] == k]['hard_case_pct'].iloc[0]) for k in ks]
    tr = [float(df_ksum[df_ksum['K'] == k]['total_reduction_pct'].iloc[0]) for k in ks]
    ax.bar(x - w/2, hc, w, label='hard case 비율', color='#c0504d')
    ax.bar(x + w/2, tr, w, label='총 교차 감소율', color='#4f81bd')
    ax.set_xticks(x); ax.set_xticklabels([f"K={k}" for k in ks])
    ax.set_ylabel('%'); ax.set_title('(1) K별 회피 성능')
    ax.legend(); ax.grid(axis='y', alpha=0.3)

    # (2) K별 평균 도보 거리 (UX 차별화 근거)
    ax = axes[0, 1]
    dist = [float(df_ksum[df_ksum['K'] == k]['v2_dist_km_mean'].iloc[0]) for k in ks]
    ax.bar(x, dist, 0.5, color='#9bbb59')
    for xi, v in zip(x, dist):
        ax.text(xi, v, f"{v:.2f}km", ha='center', va='bottom')
    ax.set_xticks(x); ax.set_xticklabels([f"K={k}" for k in ks])
    ax.set_ylabel('평균 도보 거리 (km)'); ax.set_title('(2) K별 도보 거리')
    ax.grid(axis='y', alpha=0.3)

    # (3) K별 잔존 교차 분포
    ax = axes[1, 0]
    data = [k_frames[k][k_frames[k]['v2_cross_m'] > 0]['v2_cross_m'].values for k in ks]
    if any(len(d) for d in data):
        ax.boxplot(data, labels=[f"K={k}" for k in ks], showfliers=False)
    ax.set_ylabel('잔존 교차 (m)'); ax.set_title('(3) K별 잔존 교차 분포')
    ax.grid(axis='y', alpha=0.3)

    # (4) K별 실제 경유 개수 (후보 부족 실태)
    ax = axes[1, 1]
    eff_means, eff_short = [], []
    for k in ks:
        df = k_frames[k]
        if 'k_effective' in df.columns:
            eff_means.append(df['k_effective'].mean())
            eff_short.append((df['k_effective'] < k).mean() * 100)
        else:
            eff_means.append(k); eff_short.append(0.0)
    ax2 = ax.twinx()
    ax.bar(x, eff_means, 0.5, color='#8064a2', label='실제 경유 개수(평균)')
    ax2.plot(x, eff_short, 'o--', color='#c0504d', label='요청 미달 비율(%)')
    ax.set_xticks(x); ax.set_xticklabels([f"K={k}" for k in ks])
    ax.set_ylabel('실제 경유 개수'); ax2.set_ylabel('요청 미달 비율 (%)')
    ax.set_title('(4) 후보 부족 실태')
    ax.grid(axis='y', alpha=0.3)
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc='upper left')

    fig.tight_layout()
    chart5 = V5_OUTPUT_DIR / 'step6_k_sweep_charts.png'
    fig.savefig(chart5, dpi=120, bbox_inches='tight')
    print(f"[차트 저장] {chart5.resolve()}")
    plt.show()


In [ ]:
# ==========================================================================
# 셀 19: 대표 OD 경로 비교 지도 (K별 경로 중첩)
# ==========================================================================
# 동일 OD에 대해 K를 달리했을 때 경로가 어떻게 달라지는지 시각화한다.
# 대표 OD는 가용한 K 전체에서 결과가 존재하는 OD 중, K=3 기준 교차가 큰 것을 택한다.

if not k_frames:
    print("[지도 생략] 집계 결과가 없다.")
    m5 = None
else:
    common = set(k_frames[min(k_frames)]['od_id'])
    for df in k_frames.values():
        common &= set(df['od_id'])
    base_df = k_frames[min(k_frames)]
    cand = base_df[base_df['od_id'].isin(common) & (base_df['orig_cross_m'] > 0)]

    if len(cand) == 0:
        print("[지도 생략] 공통 OD 중 교차 발생 건이 없다.")
        m5 = None
    else:
        rep_id5 = cand.sort_values('orig_cross_m', ascending=False).iloc[0]['od_id']
        rep_od5 = next(od for od in REALISTIC_OD if od['od_id'] == rep_id5)
        O, D, cands = rep_od5['O'], rep_od5['D'], rep_od5['candidates']
        print(f"[대표 OD] {rep_id5} (군구 {rep_od5['district']}, "
              f"직선 {rep_od5['direct_km']:.2f}km, 후보 {rep_od5['n_candidates']}개)")

        center = [(O[0] + D[0]) / 2, (O[1] + D[1]) / 2]
        m5 = folium.Map(location=center, zoom_start=14, tiles='CartoDB positron')

        lat_lo, lat_hi = min(O[0], D[0]) - 0.02, max(O[0], D[0]) + 0.02
        lon_lo, lon_hi = min(O[1], D[1]) - 0.02, max(O[1], D[1]) + 0.02
        for p in avoid_polygons:
            minx, miny, maxx, maxy = p['geom'].bounds
            if maxx < lon_lo or minx > lon_hi or maxy < lat_lo or miny > lat_hi:
                continue
            geom = p['geom']
            polys = [geom] if geom.geom_type == 'Polygon' else list(geom.geoms)
            for poly in polys:
                coords = [[y, x] for x, y in poly.exterior.coords]
                folium.Polygon(coords, color='#d9534f', weight=1, fill=True,
                               fill_opacity=0.25, popup=str(p.get('uname', ''))).add_to(m5)

        # K별 경로를 색을 달리해 중첩 (대표 OD 1개만 재계산 — 소수 호출)
        colors = {3: '#7f7f7f', 4: '#2b7bba', 5: '#f79646'}
        for k in sorted(k_frames):
            res_k = algorithm_greedy_with_avoidance(
                cands, O, D, avoid_union, avoid_union_prepared, avoid_union_by_code,
                gamma=GAMMA_FIXED, k=k)
            path_k = res_k.get('final_polyline') if res_k else None
            if not path_k:
                continue
            fi = res_k['final_intersection']
            cross = fi['intersection_length_m'] if (fi and fi['crosses']) else 0.0
            folium.PolyLine([[la, lo] for la, lo in path_k], color=colors.get(k, '#333'),
                            weight=4, opacity=0.85,
                            popup=f"K={k} | 교차 {cross:.0f}m | "
                                  f"{res_k['final_distance_m']/1000:.2f}km").add_to(m5)
            print(f"  K={k}: 교차 {cross:>7.0f}m | 도보 {res_k['final_distance_m']/1000:.2f}km "
                  f"| 경유 {len(res_k['final_selection'])}개")

        folium.Marker([O[0], O[1]], popup='출발(O)', icon=folium.Icon(color='green')).add_to(m5)
        folium.Marker([D[0], D[1]], popup='도착(D)', icon=folium.Icon(color='red')).add_to(m5)

        map5 = V5_OUTPUT_DIR / f'step6_k_sweep_rep_route_{rep_id5}.html'
        m5.save(str(map5))
        print(f"[지도 저장] {map5.resolve()}")
        print("  회색=K3 / 파랑=K4 / 주황=K5")

m5
